In [4]:
# Libraries

import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report


In [6]:
# === LOAD DATA ===
dataset = "dataset_rgb_96_binary_balanced.npz"
data = np.load(dataset, allow_pickle=True)
X_train = data["X_train"]
y_train = data["y_train"]

print(f"Loaded dataset with {len(X_train)} training samples.")

Loaded dataset with 2501 training samples.


In [ ]:
# === LOAD TRAINED MODEL ===
model = tf.keras.models.load_model("mobilenetv2_96_best_model.h5")

# === REPRESENTATIVE DATASET GENERATOR ===
# The generator must produce preprocessed data to match the training pipeline.
def representative_data_gen():
    """Generator for the representative dataset with correct preprocessing."""
    for i in range(2000):
        # Get a raw image from the training set
        img = X_train[i]
        # Expand dimensions to (1, 96, 96, 3) and convert to float32
        img = np.expand_dims(img, axis=0).astype(np.float32)
        
        # === IMPORTANT FIX ===
        # Apply the same preprocessing function used during training.
        # This function normalizes the pixel values from [0, 255] to [-1, 1].
        preprocessed_img = preprocess_input(img)
        
        yield [preprocessed_img]

# === CONVERTER SETTINGS FOR OPENMV ===
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Calibration dataset for quantization
converter.representative_dataset = representative_data_gen

# Force FULL int8 quantization
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# === PERFORM CONVERSION ===
tflite_model = converter.convert()

# === SAVE FILE ===
with open("mobilenetv2_96_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ Saved: mobilenetv2_224_int8_corrected.tflite (INT8 I/O, OpenMV compatible)")

INFO:tensorflow:Assets written to: C:\Users\user\AppData\Local\Temp\tmplo3wb9hf\assets


INFO:tensorflow:Assets written to: C:\Users\user\AppData\Local\Temp\tmplo3wb9hf\assets


Saved artifact at 'C:\Users\user\AppData\Local\Temp\tmplo3wb9hf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='input_layer_8')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2364906847696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906846928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906847120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906845392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906846352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906847888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906846736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906847312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906845776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906846160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2364906

c:\Users\user\anaconda3\Lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


✅ Saved: mobilenetv2_224_int8_corrected.tflite (INT8 I/O, OpenMV compatible)
